In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import random
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from models_gat import HeteroGAT

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE
# -------------------------
# Reproducibility
# -------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [2]:
# Notebook-safe paths
ROOT = Path.cwd().parent

DATA_DIR = ROOT / "data" / "data_cleaned"
GRAPH_PATH = ROOT / "outputs" / "data.pt"
OUT_DIR = ROOT / "outputs"
OUT_DIR.mkdir(exist_ok=True)

print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)
print("GRAPH_PATH:", GRAPH_PATH)
# Target relation (circRNA–disease)
pos_cd = pd.read_csv(DATA_DIR / "circRNA_disease_edges.csv")
neg_cd = pd.read_csv(DATA_DIR / "circRNA_disease_neg_edges.csv")

pos_cd['label'] = 1
neg_cd['label'] = 0

cd_edges = pd.concat([pos_cd, neg_cd], ignore_index=True)
cd_edges = cd_edges.sample(frac=1, random_state=42).reset_index(drop=True)

# Context relations
cm_edges = pd.read_csv(DATA_DIR / "circRNA_miRNA_edges.csv")
md_edges = pd.read_csv(DATA_DIR / "miRNA_disease_edges.csv")

ROOT: C:\Users\ayish\OneDrive\Documents\circRNA-disease-gnn
DATA_DIR: C:\Users\ayish\OneDrive\Documents\circRNA-disease-gnn\data\data_cleaned
GRAPH_PATH: C:\Users\ayish\OneDrive\Documents\circRNA-disease-gnn\outputs\data.pt


In [3]:
def create_kfold_splits(edges_df, n_splits=5, seed=42):

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed
    )

    folds = []

    X = edges_df[['circRNA', 'disease']]
    y = edges_df['label']

    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y)):

        train_edges = edges_df.iloc[train_idx].reset_index(drop=True)
        test_edges  = edges_df.iloc[test_idx].reset_index(drop=True)

        folds.append({
            "fold": fold_idx,
            "train_cd": train_edges,
            "test_cd": test_edges
        })

        print(f"Fold {fold_idx+1} | Train: {len(train_edges)} | Test: {len(test_edges)}")

    return folds

folds = create_kfold_splits(cd_edges)

Fold 1 | Train: 1576 | Test: 394
Fold 2 | Train: 1576 | Test: 394
Fold 3 | Train: 1576 | Test: 394
Fold 4 | Train: 1576 | Test: 394
Fold 5 | Train: 1576 | Test: 394


In [4]:
from sklearn.preprocessing import LabelEncoder

le_circ = LabelEncoder()
le_mir  = LabelEncoder()
le_dis  = LabelEncoder()

le_circ.fit(pd.concat([cm_edges["circRNA"], cd_edges["circRNA"]], ignore_index=True).astype(str))
le_mir.fit(pd.concat([cm_edges["miRNA"], md_edges["miRNA"]], ignore_index=True).astype(str))
le_dis.fit(pd.concat([md_edges["disease"], cd_edges["disease"]], ignore_index=True).astype(str))

num_circ = len(le_circ.classes_)
num_mir  = len(le_mir.classes_)
num_dis  = len(le_dis.classes_)

print(num_circ, num_mir, num_dis)

828 521 122


In [5]:
from torch_geometric.data import HeteroData
from torch_geometric.transforms import ToUndirected
import networkx as nx
import torch

def build_graph(train_cd, cm_edges, md_edges,
                num_circ, num_mir, num_dis,
                le_circ, le_mir, le_dis):

    data = HeteroData()

    # -------------------------------------------------
    # Encode edge indices
    # -------------------------------------------------

    # circRNA–miRNA (fixed)
    cm_src = torch.tensor(le_circ.transform(cm_edges["circRNA"]))
    cm_dst = torch.tensor(le_mir.transform(cm_edges["miRNA"]))

    # miRNA–disease (fixed)
    md_src = torch.tensor(le_mir.transform(md_edges["miRNA"]))
    md_dst = torch.tensor(le_dis.transform(md_edges["disease"]))

    # circRNA–disease (TRAIN ONLY)
    train_cd_pos = train_cd[train_cd["label"] == 1]

    cd_src = torch.tensor(le_circ.transform(train_cd_pos["circRNA"]))
    cd_dst = torch.tensor(le_dis.transform(train_cd_pos["disease"]))

    # -------------------------------------------------
    # Add edges to HeteroData
    # -------------------------------------------------

    data["circRNA", "interacts", "miRNA"].edge_index = torch.stack([cm_src, cm_dst])
    data["miRNA", "interacts", "disease"].edge_index = torch.stack([md_src, md_dst])
    data["circRNA", "associated", "disease"].edge_index = torch.stack([cd_src, cd_dst])

    # -------------------------------------------------
    # Build NetworkX graph (for betweenness)
    # -------------------------------------------------

    G = nx.Graph()
    G.add_nodes_from(range(num_circ + num_mir + num_dis))

    circ_offset = 0
    mir_offset  = num_circ
    dis_offset  = num_circ + num_mir

    # circRNA–miRNA
    G.add_edges_from(
        [(circ_offset + s.item(), mir_offset + d.item())
         for s, d in zip(cm_src, cm_dst)]
    )

    # miRNA–disease
    G.add_edges_from(
        [(mir_offset + s.item(), dis_offset + d.item())
         for s, d in zip(md_src, md_dst)]
    )

    # circRNA–disease (TRAIN ONLY)
    G.add_edges_from(
        [(circ_offset + s.item(), dis_offset + d.item())
         for s, d in zip(cd_src, cd_dst)]
    )

    # -------------------------------------------------
    # Betweenness centrality (per fold!)
    # -------------------------------------------------

    bc = nx.betweenness_centrality(G, normalized=True)

    circ_bc = torch.tensor(
        [bc[circ_offset + i] for i in range(num_circ)],
        dtype=torch.float32
    ).unsqueeze(1)

    mir_bc = torch.tensor(
        [bc[mir_offset + i] for i in range(num_mir)],
        dtype=torch.float32
    ).unsqueeze(1)

    dis_bc = torch.tensor(
        [bc[dis_offset + i] for i in range(num_dis)],
        dtype=torch.float32
    ).unsqueeze(1)

    def normalize(x):
        return (x - x.mean()) / (x.std() + 1e-8)

    circ_bc = normalize(circ_bc)
    mir_bc  = normalize(mir_bc)
    dis_bc  = normalize(dis_bc)

    # -------------------------------------------------
    # Degree features (recomputed per fold)
    # -------------------------------------------------

    circ_deg = torch.zeros((num_circ, 1))
    mir_deg  = torch.zeros((num_mir, 1))
    dis_deg  = torch.zeros((num_dis, 1))

    for i in cm_src: circ_deg[i] += 1
    for i in cd_src: circ_deg[i] += 1

    for i in cm_dst: mir_deg[i] += 1
    for i in md_src: mir_deg[i] += 1

    for i in md_dst: dis_deg[i] += 1
    for i in cd_dst: dis_deg[i] += 1

    # -------------------------------------------------
    # Node-type one-hot
    # -------------------------------------------------

    circ_type = torch.tensor([[1, 0, 0]]).repeat(num_circ, 1)
    mir_type  = torch.tensor([[0, 1, 0]]).repeat(num_mir, 1)
    dis_type  = torch.tensor([[0, 0, 1]]).repeat(num_dis, 1)

    # -------------------------------------------------
    # Final node features
    # -------------------------------------------------

    data["circRNA"].x = torch.cat(
        [circ_deg, torch.log1p(circ_deg), circ_bc, circ_type], dim=1
    ).float()

    data["miRNA"].x = torch.cat(
        [mir_deg, torch.log1p(mir_deg), mir_bc, mir_type], dim=1
    ).float()

    data["disease"].x = torch.cat(
        [dis_deg, torch.log1p(dis_deg), dis_bc, dis_type], dim=1
    ).float()

    # -------------------------------------------------
    # Make undirected
    # -------------------------------------------------

    data = ToUndirected()(data)

    return data

In [6]:
for fold_data in folds:

    train_cd = fold_data["train_cd"]
    test_cd  = fold_data["test_cd"]

    data = build_graph(
        train_cd, cm_edges, md_edges,
        num_circ, num_mir, num_dis,
        le_circ, le_mir, le_dis
    )

    # train model on data
    # evaluate on test_cd

In [7]:
results = []
EPOCHS = 50

for fold_data in folds:

    fold_id = fold_data["fold"]
    print(f"\n==============================")
    print(f"Running GAT Fold {fold_id + 1}")
    print(f"==============================")

    set_seed(42)

    # ----------------------------
    # Split train into train/val
    # ----------------------------
    train_cd_full = fold_data["train_cd"]
    test_cd       = fold_data["test_cd"]

    val_split = int(0.9 * len(train_cd_full))
    train_cd = train_cd_full.iloc[:val_split]
    val_cd   = train_cd_full.iloc[val_split:]

    # ----------------------------
    # Build graph using TRAIN CD only
    # ----------------------------
    data = build_graph(
        train_cd, cm_edges, md_edges,
        num_circ, num_mir, num_dis,
        le_circ, le_mir, le_dis
    ).to(DEVICE)

    # ----------------------------
    # Prepare edges
    # ----------------------------
    def prepare_edges(df):
        src = torch.tensor(le_circ.transform(df["circRNA"])).to(DEVICE)
        dst = torch.tensor(le_dis.transform(df["disease"])).to(DEVICE)
        labels = torch.tensor(df["label"].values).float().to(DEVICE)
        return src, dst, labels

    train_src, train_dst, train_labels = prepare_edges(train_cd)
    val_src, val_dst, val_labels       = prepare_edges(val_cd)
    test_src, test_dst, test_labels    = prepare_edges(test_cd)

    # ----------------------------
    # Initialize GAT
    # ----------------------------
    model = HeteroGAT(
        in_channels=data["circRNA"].x.size(1),
        hidden_channels=64,
        out_channels=64,
        heads=4,
        dropout=0.2
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn   = nn.BCEWithLogitsLoss()

    best_val_aupr = 0.0

    # ----------------------------
    # Training loop
    # ----------------------------
    for epoch in range(1, EPOCHS + 1):

        model.train()
        optimizer.zero_grad()

        emb = model(data.x_dict, data.edge_index_dict)
        circ_emb, dis_emb = emb["circRNA"], emb["disease"]

        logits = (circ_emb[train_src] * dis_emb[train_dst]).sum(dim=1)
        loss = loss_fn(logits, train_labels)

        loss.backward()
        optimizer.step()

        # -------- Validation --------
        model.eval()
        with torch.no_grad():

            emb = model(data.x_dict, data.edge_index_dict)
            circ_emb, dis_emb = emb["circRNA"], emb["disease"]

            val_logits = (circ_emb[val_src] * dis_emb[val_dst]).sum(dim=1)
            val_scores = torch.sigmoid(val_logits).cpu().numpy()
            val_true   = val_labels.cpu().numpy()

            val_auc  = roc_auc_score(val_true, val_scores)
            val_aupr = average_precision_score(val_true, val_scores)

        print(
            f"Fold {fold_id+1} | "
            f"Epoch {epoch:03d} | "
            f"Loss {loss.item():.4f} | "
            f"Val AUC {val_auc:.4f} | "
            f"Val AUPR {val_aupr:.4f}"
        )

        if val_aupr > best_val_aupr:
            best_val_aupr = val_aupr
            torch.save(
                model.state_dict(),
                OUT_DIR / f"gat_best_fold{fold_id}.pth"
            )

    # ----------------------------
    # Test evaluation
    # ----------------------------
    model.load_state_dict(
        torch.load(
            OUT_DIR / f"gat_best_fold{fold_id}.pth",
            map_location=DEVICE
        )
    )

    model.eval()
    with torch.no_grad():

        emb = model(data.x_dict, data.edge_index_dict)
        circ_emb, dis_emb = emb["circRNA"], emb["disease"]

        test_logits = (circ_emb[test_src] * dis_emb[test_dst]).sum(dim=1)
        test_scores = torch.sigmoid(test_logits).cpu().numpy()
        test_true   = test_labels.cpu().numpy()

        test_auc  = roc_auc_score(test_true, test_scores)
        test_aupr = average_precision_score(test_true, test_scores)

    print(f"Fold {fold_id+1} | Test AUC {test_auc:.4f} | Test AUPR {test_aupr:.4f}")

    results.append((fold_id, test_auc, test_aupr))

# ----------------------------
# Final Results
# ----------------------------
results_df = pd.DataFrame(results, columns=["fold", "auc", "aupr"])

print("\n===== FINAL 5-FOLD RESULTS (GAT) =====")
print(results_df)
print("\nMean ± Std")
print(results_df[["auc", "aupr"]].agg(["mean", "std"]))


Running GAT Fold 1
Fold 1 | Epoch 001 | Loss 0.7043 | Val AUC 0.3209 | Val AUPR 0.3873
Fold 1 | Epoch 002 | Loss 0.6713 | Val AUC 0.4121 | Val AUPR 0.4276
Fold 1 | Epoch 003 | Loss 0.6492 | Val AUC 0.4672 | Val AUPR 0.4601
Fold 1 | Epoch 004 | Loss 0.6378 | Val AUC 0.5094 | Val AUPR 0.5039
Fold 1 | Epoch 005 | Loss 0.6273 | Val AUC 0.5530 | Val AUPR 0.5719
Fold 1 | Epoch 006 | Loss 0.6185 | Val AUC 0.5941 | Val AUPR 0.6365
Fold 1 | Epoch 007 | Loss 0.6069 | Val AUC 0.6303 | Val AUPR 0.6722
Fold 1 | Epoch 008 | Loss 0.6000 | Val AUC 0.6376 | Val AUPR 0.6835
Fold 1 | Epoch 009 | Loss 0.5887 | Val AUC 0.6459 | Val AUPR 0.6881
Fold 1 | Epoch 010 | Loss 0.5829 | Val AUC 0.6815 | Val AUPR 0.6988
Fold 1 | Epoch 011 | Loss 0.5805 | Val AUC 0.7050 | Val AUPR 0.7187
Fold 1 | Epoch 012 | Loss 0.5713 | Val AUC 0.7295 | Val AUPR 0.7343
Fold 1 | Epoch 013 | Loss 0.5646 | Val AUC 0.7484 | Val AUPR 0.7510
Fold 1 | Epoch 014 | Loss 0.5631 | Val AUC 0.7606 | Val AUPR 0.7603
Fold 1 | Epoch 015 | Loss 0.